# Experiments: Measuring the Impact of Class Imbalance

We compare the baseline with `class_weight='balanced'` using the same stratified split. The decision is not based on accuracy alone: we also measure macro F1 and recall for `functional needs repair`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.metrics import recall_score

PROJECT_ROOT = None
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / 'src').is_dir() and (candidate / 'data').is_dir():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the project root containing src/ and data/.')
sys.path.insert(0, str(PROJECT_ROOT))

from src.data import load_pump_it_up_data
from src.modeling import build_random_forest_pipeline, classification_metrics, stratified_split

train_values, train_labels, _ = load_pump_it_up_data(PROJECT_ROOT / 'data')
train = train_values.merge(train_labels, on='id', validate='one_to_one')
X = train.drop(columns=['status_group'])
y = train['status_group']
X_train, X_validation, y_train, y_validation = stratified_split(X, y)

In [ ]:
rows = []
for name, class_weight in [('baseline', None), ('balanced', 'balanced')]:
    pipeline = build_random_forest_pipeline(
        X_train, n_estimators=50, class_weight=class_weight, random_state=42
    )
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_validation)
    metrics = classification_metrics(y_validation, predictions)
    repair_recall = recall_score(
        y_validation, predictions, labels=['functional needs repair'], average=None
    )[0]
    rows.append({
        'model': name,
        'accuracy': metrics['accuracy'],
        'macro_f1': metrics['macro_f1'],
        'repair_recall': repair_recall,
    })

comparison = pd.DataFrame(rows).round(4)
display(comparison)

## Provisional decision

The balanced model sacrifices some accuracy but improves macro F1 and recall for the class that needs repair. For a maintenance use case, this trade-off is defensible and should be compared with the operational cost of each error before selecting the final model.